# Dynamic Agents in LangChain (Summary)

## 📌 Core Idea
In real-world applications, users are not the same:
- Different languages (English, French, etc.)
- Different roles (internal vs external)
- Different access levels and needs

➡️ A static "one-size-fits-all" agent is not enough.

**Solution:** Build a **Dynamic Agent** that can adapt during runtime.

---

## 🔥 What is a Dynamic Agent?
An agent that can dynamically change:
- Prompt (instructions)
- Tools (what it can use)
- Model (which LLM it runs on)

➡️ These changes can happen **during the conversation**, even multiple times.

---

## 🧠 How It Works: Middleware

### 1. Node-Style Middleware
- Works with: `state` and `runtime`
- Used for:
  - Modifying messages
  - Trimming conversation history

❗ Limitation: Cannot modify the model itself

---

### 2. Wrap-Style Middleware (Key Concept 🔥)
- Works with: `ModelRequest`
- Gives full control over:
  - System prompt
  - Tools
  - Model
  - State

➡️ You are directly modifying how the model behaves

---

## 📦 ModelRequest
Contains:
- `system_prompt`
- `tools`
- `model`
- `state`

➡️ Modifying this = changing the agent’s behavior

---

## 🧪 Key Examples

### 1. Dynamic Language Switching
- Detect user's preferred language
- Switch prompt accordingly

➡️ Example:
- English user → respond in English
- French user → respond in French

---

### 2. Dynamic Tools Based on User Role
- Internal user:
  - Access to database + web search
- External user:
  - Web search only

➡️ Middleware modifies:
```python
model_request.tools
```
---
### 3. Dynamic Model Switching

- Short conversation → small / cheap model  
- Long conversation → larger / more powerful model  

### Example Logic

```python
if message_count > 10:
    use_large_model()
else:
    use_small_model()

In [7]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "dynamic models"

In [8]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00


In [9]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# dynamic prompts

In [4]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [5]:
@dataclass
class LanguageContext:
    user_language: str = "English"


In [6]:
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."

    if user_language == "English":
        return base_prompt

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia duit! Tá mé go maith, go raibh maith agat. Agus tú féin?


In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

Hola, estoy bien, ¿y tú?


In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour, je vais bien, merci. Et vous ?


# dynamic tools

In [2]:
! pip install tavily -q

  Preparing metadata (setup.py) ... done


In [4]:
! pip install langchain_community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

In [13]:
tavily = TavilyClient(api_key=userdata.get('tavily'))

db = SQLDatabase.from_uri("sqlite:////content/Chinook.db")

In [14]:
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [15]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [16]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools)

    return handler(request)

In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [19]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

{'type': 'text', 'text': 'I am sorry, I cannot answer this question. I do not have access to a database of artists.', 'extras': {'signature': 'Ct4EAb4+9vvi16UoxQYjM+NpC528XuUpwc24i5i3mOqohDSvqxFkJjpNEFP4TVap3VcWO+uNTdjAMv3NK+WoYpehVxcPjeTooU6G9GxE4e0Nb1JLZn66eukOOeA4EvE4Vqd8aqKP1YGGI50ohvq/T6XNluO8xDkoHQfOAiDWstfqp0RNT0NY8vuBrOJZ6AlkUFR4QECw5kiODXI7zUvMPeblINlUxn5vOzX0Anebxl180pjZbuqGM1Q3LtX4C/FEu5BY5oTldOTYIVtF+4OTOEmDjcNZXX//8W+aO93moY+aXAYyy0ohj1m51Vv6wcqyUeoX2F8BUmXrtPOv6GbeqjV9cirUMUy0emE/eJjTlRlABV8cMDX/sCbTH+wyMXWl28c6CISLE4wgt2t6GzrGDMLgkNV0mMdFjz8f+TlNoOUcSIEJ8YJ0HtQ+Nib1Z9JwlWslJmxkvv1Y+/dXF31qnX2WFWL2wunmACl931UbyaljwiB5dmi9fNf2Qnm4nIRBa35kw2Gu7cCxw10gKot6MW+eEz983yYC9DtQ9kV1Kilsp6Lg1sH52gJ/nFy4G8KulmMeIZiQAH9cn1uTFelzbSW/Ty8fNxDuEm3cm1o1ub45WA+i9C9wBlBPHsY+/BgMRWrPNQAa+aryhuy4so2iKXL3U4JVPzY+ps1IkSHRjJpmN30Ks05/OAU5BlvLhLvb5r7NH4hm0XNIp3PqicqYwYTrI8rNUY9VdNKEi3I9eZjux06lWdVQdmePpzCebj65HhyF2O30zrEhKatR9bLzcB9H3T0KoqdaIH9gNulHGk8W'}}


In [20]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'There are 275 artists in the database.', 'extras': {'signature': 'Ct0BAb4+9vuViPI5aW/cGegSvEu1YP7q9qMjsei6O73DEtYJhOd0c7yoOCx6Fw/A4JVVQp7gdnlywIaaLnHavhdGrOtcCAI28RXTxiIV4zV9KhhNMc7UGwdVSWctCz8NmfoVa2GRnHlcHI/AR37eoGSUouZdoVXO+9yV9huArLdBzf/1SB4bvEWPSO9pQRuvZNCQ2T7QEvlvXCm/xEVRWtpWGHAmezMQq/tJO5Ob007B68V17J/VwXXGshkufjvehJw+qManOKO0TKqZ3G8MdDdGXY2R2tEZba9McMkXZKk='}}]


# dynamic models

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])

In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])